# Advanced Configuration Guide

This notebook covers advanced configuration options for MetaPathPredict, including:

1. Configuration system with Pydantic v2
2. Training hyperparameters
3. Model architecture customization
4. Data pipeline configuration
5. Experiment tracking setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Literal

## 1. Configuration System

MetaPathPredict uses Pydantic v2 for type-safe configuration with validation.

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Literal


class ModelConfig(BaseModel):
    """Model architecture configuration."""
    
    # Architecture type
    architecture: Literal["cnn", "contrastive", "reinforcement"] = "cnn"
    
    # CNN settings
    kernel_preset: Literal["small", "medium", "large"] = "medium"
    hidden_channels: List[int] = Field(default=[32, 64, 128])
    dropout: float = Field(default=0.3, ge=0.0, le=0.9)
    
    # Contrastive settings
    projection_dim: int = Field(default=128, ge=32)
    temperature: float = Field(default=0.5, gt=0.0)
    
    # RL settings
    rl_algorithm: Literal["dqn", "reinforce", "a2c"] = "reinforce"
    hidden_dim: int = Field(default=256, ge=64)
    
    @field_validator('hidden_channels')
    @classmethod
    def validate_channels(cls, v):
        if len(v) < 1:
            raise ValueError("At least one hidden channel required")
        if not all(c > 0 for c in v):
            raise ValueError("All channels must be positive")
        return v


class TrainingConfig(BaseModel):
    """Training hyperparameters."""
    
    # Basic settings
    epochs: int = Field(default=100, ge=1)
    batch_size: int = Field(default=32, ge=1)
    learning_rate: float = Field(default=1e-3, gt=0.0)
    weight_decay: float = Field(default=1e-4, ge=0.0)
    
    # Scheduler
    scheduler: Literal["cosine", "step", "plateau", "none"] = "cosine"
    warmup_epochs: int = Field(default=5, ge=0)
    min_lr: float = Field(default=1e-6, ge=0.0)
    
    # Mixed precision
    use_amp: bool = True
    
    # Early stopping
    early_stopping: bool = True
    patience: int = Field(default=10, ge=1)
    
    # Gradient settings
    gradient_clip: Optional[float] = Field(default=1.0, ge=0.0)
    accumulation_steps: int = Field(default=1, ge=1)
    
    @model_validator(mode='after')
    def validate_warmup(self):
        if self.warmup_epochs >= self.epochs:
            raise ValueError("warmup_epochs must be less than epochs")
        return self


class DataConfig(BaseModel):
    """Data pipeline configuration."""
    
    # Sequence settings
    max_length: int = Field(default=500, ge=100, le=10000)
    min_length: int = Field(default=100, ge=50)
    
    # Preprocessing
    encoding: Literal["onehot", "kmer"] = "onehot"
    kmer_size: int = Field(default=3, ge=1, le=7)
    
    # Augmentation
    augmentation: bool = True
    aug_crop_ratio: float = Field(default=0.9, gt=0.5, le=1.0)
    aug_mask_ratio: float = Field(default=0.1, ge=0.0, le=0.3)
    aug_noise_std: float = Field(default=0.1, ge=0.0)
    
    # Data loading
    num_workers: int = Field(default=4, ge=0)
    pin_memory: bool = True
    
    # Split ratios
    train_ratio: float = Field(default=0.8, gt=0.0, lt=1.0)
    val_ratio: float = Field(default=0.1, gt=0.0, lt=1.0)
    test_ratio: float = Field(default=0.1, gt=0.0, lt=1.0)
    
    @model_validator(mode='after')
    def validate_ratios(self):
        total = self.train_ratio + self.val_ratio + self.test_ratio
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Ratios must sum to 1.0, got {total}")
        return self


class ExperimentConfig(BaseModel):
    """Experiment tracking configuration."""
    
    # Tracker type
    tracker: Literal["mlflow", "wandb", "duckdb", "none"] = "none"
    experiment_name: str = "metapathpredict"
    run_name: Optional[str] = None
    tags: Dict[str, str] = Field(default_factory=dict)
    
    # MLflow settings
    mlflow_tracking_uri: str = "mlruns"
    
    # W&B settings
    wandb_project: str = "metapathpredict"
    wandb_entity: Optional[str] = None
    wandb_mode: Literal["online", "offline", "disabled"] = "online"
    
    # DuckDB settings
    duckdb_path: str = "experiments.duckdb"


class Config(BaseModel):
    """Complete configuration."""
    
    model: ModelConfig = Field(default_factory=ModelConfig)
    training: TrainingConfig = Field(default_factory=TrainingConfig)
    data: DataConfig = Field(default_factory=DataConfig)
    experiment: ExperimentConfig = Field(default_factory=ExperimentConfig)
    
    # Paths
    data_dir: str = "data"
    output_dir: str = "outputs"
    checkpoint_dir: str = "checkpoints"
    
    # Reproducibility
    seed: int = Field(default=42)
    deterministic: bool = False
    
    @classmethod
    def from_yaml(cls, path: str) -> "Config":
        """Load configuration from YAML file."""
        with open(path) as f:
            data = yaml.safe_load(f)
        return cls(**data)
    
    def to_yaml(self, path: str):
        """Save configuration to YAML file."""
        with open(path, 'w') as f:
            yaml.dump(self.model_dump(), f, default_flow_style=False)
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary."""
        return self.model_dump()

In [ ]:
# Create default configuration
config = Config()
print("Default Configuration:")
print(yaml.dump(config.to_dict(), default_flow_style=False))

In [ ]:
# Create custom configuration
custom_config = Config(
    model=ModelConfig(
        architecture="contrastive",
        kernel_preset="large",
        hidden_channels=[64, 128, 256],
        projection_dim=256,
    ),
    training=TrainingConfig(
        epochs=200,
        batch_size=64,
        learning_rate=3e-4,
        scheduler="cosine",
        warmup_epochs=10,
    ),
    data=DataConfig(
        max_length=1000,
        augmentation=True,
    ),
    experiment=ExperimentConfig(
        tracker="mlflow",
        experiment_name="contrastive_exp",
    ),
    seed=123,
)

print("Custom Configuration:")
print(yaml.dump(custom_config.to_dict(), default_flow_style=False))

## 2. Configuration Validation

In [ ]:
from pydantic import ValidationError

# Test validation - invalid learning rate
try:
    invalid_config = TrainingConfig(learning_rate=-0.001)
except ValidationError as e:
    print("Validation Error (negative learning rate):")
    print(e)

In [ ]:
# Test validation - invalid split ratios
try:
    invalid_config = DataConfig(
        train_ratio=0.8,
        val_ratio=0.2,
        test_ratio=0.2,  # Sum > 1.0
    )
except ValidationError as e:
    print("Validation Error (invalid ratios):")
    print(e)

In [ ]:
# Test validation - warmup > epochs
try:
    invalid_config = TrainingConfig(
        epochs=10,
        warmup_epochs=15,
    )
except ValidationError as e:
    print("Validation Error (warmup > epochs):")
    print(e)

## 3. Example Configurations

Here are some preset configurations for common use cases.

In [ ]:
# Quick training (for testing)
quick_config = Config(
    model=ModelConfig(
        kernel_preset="small",
        hidden_channels=[16, 32],
    ),
    training=TrainingConfig(
        epochs=5,
        batch_size=64,
        early_stopping=False,
    ),
    data=DataConfig(
        max_length=200,
        augmentation=False,
    ),
)

print("Quick Training Config:")
print(f"  Epochs: {quick_config.training.epochs}")
print(f"  Model channels: {quick_config.model.hidden_channels}")

In [ ]:
# Production training
production_config = Config(
    model=ModelConfig(
        architecture="cnn",
        kernel_preset="medium",
        hidden_channels=[64, 128, 256, 512],
        dropout=0.4,
    ),
    training=TrainingConfig(
        epochs=200,
        batch_size=32,
        learning_rate=1e-3,
        scheduler="cosine",
        warmup_epochs=10,
        early_stopping=True,
        patience=20,
        use_amp=True,
    ),
    data=DataConfig(
        max_length=1000,
        augmentation=True,
        num_workers=8,
    ),
    experiment=ExperimentConfig(
        tracker="mlflow",
        experiment_name="production_run",
    ),
    deterministic=True,
)

print("Production Config:")
print(f"  Epochs: {production_config.training.epochs}")
print(f"  Model channels: {production_config.model.hidden_channels}")
print(f"  Tracker: {production_config.experiment.tracker}")

In [ ]:
# Contrastive pretraining config
contrastive_config = Config(
    model=ModelConfig(
        architecture="contrastive",
        kernel_preset="large",
        hidden_channels=[64, 128, 256],
        projection_dim=256,
        temperature=0.07,
    ),
    training=TrainingConfig(
        epochs=100,
        batch_size=256,  # Large batch for contrastive
        learning_rate=3e-4,
        scheduler="cosine",
        warmup_epochs=10,
    ),
    data=DataConfig(
        augmentation=True,
        aug_crop_ratio=0.85,
        aug_mask_ratio=0.15,
    ),
)

print("Contrastive Config:")
print(f"  Projection dim: {contrastive_config.model.projection_dim}")
print(f"  Temperature: {contrastive_config.model.temperature}")
print(f"  Batch size: {contrastive_config.training.batch_size}")

In [ ]:
# RL training config
rl_config = Config(
    model=ModelConfig(
        architecture="reinforcement",
        rl_algorithm="a2c",
        hidden_dim=512,
    ),
    training=TrainingConfig(
        epochs=1000,  # Episodes for RL
        batch_size=32,
        learning_rate=7e-4,
    ),
)

print("RL Config:")
print(f"  Algorithm: {rl_config.model.rl_algorithm}")
print(f"  Hidden dim: {rl_config.model.hidden_dim}")

## 4. Saving and Loading Configurations

In [ ]:
# Save configuration to YAML
config_path = "../configs/example_config.yaml"
production_config.to_yaml(config_path)
print(f"Configuration saved to {config_path}")

# Load configuration from YAML
loaded_config = Config.from_yaml(config_path)
print(f"\nLoaded configuration:")
print(f"  Epochs: {loaded_config.training.epochs}")
print(f"  Architecture: {loaded_config.model.architecture}")

## 5. Using Configuration in Training

In [ ]:
import torch
import torch.nn as nn

def create_model_from_config(config: Config) -> nn.Module:
    """
    Create model based on configuration.
    """
    model_config = config.model
    
    if model_config.architecture == "cnn":
        # Import your actual model
        # from metapathpredict.models import ConfigurableCNN
        # return ConfigurableCNN(
        #     kernel_preset=model_config.kernel_preset,
        #     hidden_channels=model_config.hidden_channels,
        #     dropout=model_config.dropout,
        # )
        print(f"Creating CNN with kernel={model_config.kernel_preset}")
        return nn.Identity()  # Placeholder
    
    elif model_config.architecture == "contrastive":
        print(f"Creating Contrastive model with projection_dim={model_config.projection_dim}")
        return nn.Identity()
    
    elif model_config.architecture == "reinforcement":
        print(f"Creating RL agent with algorithm={model_config.rl_algorithm}")
        return nn.Identity()
    
    else:
        raise ValueError(f"Unknown architecture: {model_config.architecture}")


def create_optimizer_from_config(model: nn.Module, config: Config) -> torch.optim.Optimizer:
    """
    Create optimizer based on configuration.
    """
    training_config = config.training
    
    return torch.optim.AdamW(
        model.parameters(),
        lr=training_config.learning_rate,
        weight_decay=training_config.weight_decay,
    )


def create_scheduler_from_config(
    optimizer: torch.optim.Optimizer,
    config: Config,
):
    """
    Create learning rate scheduler based on configuration.
    """
    training_config = config.training
    
    if training_config.scheduler == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=training_config.epochs - training_config.warmup_epochs,
            eta_min=training_config.min_lr,
        )
    elif training_config.scheduler == "step":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=30,
            gamma=0.1,
        )
    elif training_config.scheduler == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            patience=5,
        )
    else:
        return None

In [ ]:
# Example usage
config = production_config

model = create_model_from_config(config)
optimizer = create_optimizer_from_config(model, config)
scheduler = create_scheduler_from_config(optimizer, config)

print(f"\nOptimizer: {type(optimizer).__name__}")
print(f"Learning rate: {optimizer.param_groups[0]['lr']}")
print(f"Scheduler: {type(scheduler).__name__ if scheduler else 'None'}")

## 6. Environment Variables Override

You can override configuration values using environment variables.

In [ ]:
import os
from pydantic_settings import BaseSettings

class EnvironmentConfig(BaseSettings):
    """
    Configuration that can be overridden by environment variables.
    
    Environment variables are prefixed with METAPATH_.
    Nested values use double underscore: METAPATH_TRAINING__EPOCHS=100
    """
    
    # Override these with env vars
    seed: int = 42
    data_dir: str = "data"
    output_dir: str = "outputs"
    
    # Experiment tracking
    tracker: str = "none"
    mlflow_tracking_uri: str = "mlruns"
    wandb_project: str = "metapathpredict"
    
    class Config:
        env_prefix = "METAPATH_"


# Set environment variable
os.environ["METAPATH_SEED"] = "123"
os.environ["METAPATH_TRACKER"] = "mlflow"

# Load configuration (will use env vars)
env_config = EnvironmentConfig()
print(f"Seed from env: {env_config.seed}")
print(f"Tracker from env: {env_config.tracker}")

## Summary

Key configuration features:

1. **Type Safety**: Pydantic v2 provides automatic validation
2. **Hierarchical**: Organized by concern (model, training, data, experiment)
3. **Defaults**: Sensible defaults for quick starts
4. **Validation**: Custom validators for complex constraints
5. **Serialization**: Easy YAML/JSON import/export
6. **Environment Override**: Support for env var configuration